# Data Cleaning: The Unified Process
**Objective:** Consolidate all individual cleaning efforts into one robust, error-free pipeline. We will transform 66k raw listings into a machine-learning-ready dataset.

---

## 1. Setup & Data Loading
We start by loading our primary merged dataset from the `dataset` folder.

In [6]:
import pandas as pd
import numpy as np
import re
import os

# Locate the raw merged data
raw_path = 'dataset/ouedkniss_informatique_ordinateur_portable_20250811_180006.csv'
if not os.path.exists(raw_path):
    # Try absolute path if local fails
    raw_path = '../02_Data_Collection_Integration/dataset/ouedkniss_informatique_ordinateur_portable_20250811_180006.csv'

df = pd.read_csv(raw_path)
print(f" Loaded {len(df)} raw records.")
print(f"Initial Columns: {df.columns.tolist()}")

 Loaded 20819 raw records.
Initial Columns: ['reference', 'title', 'description', 'price_preview', 'created_at', 'city', 'spec_Carte graphique', 'spec_Etat', 'spec_Marque', 'spec_Processeur', 'spec_RAM', 'spec_Reférence carte graphique dédiée', 'spec_Reférence carte graphique integrée', 'spec_Taille du disque', 'spec_Taille écran', 'spec_Type disque']


## 2. Core Identity Standardisation
Mapping raw columns to our target features: `PRICE`, `BRAND`, `MODEL`, and `CONDITION`.

In [7]:
# 1. Select and Rename relevant columns
df = df.rename(columns={
    'price_preview': 'PRICE',
    'spec_Marque': 'LAPTOP_BRAND',
    'title': 'LAPTOP_MODEL',
    'spec_Etat': 'LAPTOP_CONDITION',
    'spec_RAM': 'RAM_SIZE',
    'spec_Taille du disque': 'STORAGE_SIZE',
    'spec_Taille écran': 'SCREEN_SIZE',
    'spec_Processeur': 'CPU',
    'spec_Type disque': 'STORAGE_TYPE'
})

# 2. Price Cleaning
df['PRICE'] = pd.to_numeric(df['PRICE'], errors='coerce')
df = df[df['PRICE'].notna()]
df = df[(df['PRICE'] > 5000) & (df['PRICE'] < 1000000)] # Filter extreme noise

# 3. Brand Standardisation
df['LAPTOP_BRAND'] = df['LAPTOP_BRAND'].str.upper().str.strip()
df['LAPTOP_BRAND'] = df['LAPTOP_BRAND'].replace({'HEWLETT PACKARD': 'HP', 'APPEL': 'APPLE', 'DELL LATITUDE': 'DELL'})

print(f" Core Identity Cleaned. Remaining records: {len(df)}")

 Core Identity Cleaned. Remaining records: 16876


## 3. Storage & Memory Engineering
Converting text-based specifications into meaningful numerical values for modeling.

In [8]:
def extract_gb(val):
    if pd.isna(val): return 0
    val = str(val).upper().replace(' ', '')
    num_match = re.findall(r'(\d+)', val)
    if not num_match: return 0
    num = int(num_match[0])
    if 'TB' in val or 'TO' in val: return num * 1024
    return num

# Extract Memory
df['RAM_GB'] = df['RAM_SIZE'].apply(extract_gb)

# Extract Storage logic (SSD vs HDD)
def distribute_storage(row):
    size = extract_gb(row['STORAGE_SIZE'])
    s_type = str(row['STORAGE_TYPE']).upper()
    
    ssd, hdd = 0, 0
    if 'SSD' in s_type and 'HDD' in s_type:
        # Complex case like 1TB+256SSD
        ssd = 256 # Default assumption if mixed
        hdd = size if size > 256 else 1024
    elif 'SSD' in s_type:
        ssd = size
    elif 'HDD' in s_type:
        hdd = size
    else:
        # Default to SSD for small sizes, HDD for large
        if size <= 512: ssd = size
        else: hdd = size
    return ssd, hdd

storage_split = df.apply(distribute_storage, axis=1)
df['SSD_GB'] = [x[0] for x in storage_split]
df['HDD_GB'] = [x[1] for x in storage_split]

print(" Storage & RAM successfully extracted and split.")

 Storage & RAM successfully extracted and split.


## 4. Screen Features & Final Logic
Cleaning screen sizes and ensuring logical consistency across features.

In [9]:
def clean_screen(val):
    if pd.isna(val): return 15.6
    match = re.findall(r'(\d+\.?\d*)', str(val))
    if not match: return 15.6
    size = float(match[0])
    if size > 100: size /= 10
    if size < 10 or size > 20: return 15.6 # Default for outliers
    return size

df['SCREEN_SIZE'] = df['SCREEN_SIZE'].apply(clean_screen)

# Cross-Feature Validation: High Price Laptops MUST have an SSD
df.loc[(df['PRICE'] > 150000) & (df['SSD_GB'] == 0), 'SSD_GB'] = 512

print(" Screen sizes cleaned and high-end outliers corrected.")

 Screen sizes cleaned and high-end outliers corrected.


## 5. Exporting Final Dataset
We save the result to the modeling folder for immediate use.

In [10]:
output_path = '../06_Model_Training_Evaluation/final_cleaned_dataset.csv'
df.to_csv(output_path, index=False)

print(f" SUCCESS! Cleaned dataset saved at: {output_path}")
print("Sample of cleaned data:")
df[['LAPTOP_BRAND', 'PRICE', 'RAM_GB', 'SSD_GB', 'SCREEN_SIZE']].head()

 SUCCESS! Cleaned dataset saved at: ../06_Model_Training_Evaluation/final_cleaned_dataset.csv
Sample of cleaned data:


,LAPTOP_BRAND,PRICE,RAM_GB,SSD_GB,SCREEN_SIZE
1,HP,65000.0,8,240,17.3
2,DELL,115000.0,16,512,14.0
3,NaN,62000.0,0,0,15.6
5,NaN,117000.0,32,512,14.0
6,DELL,59000.0,8,256,13.0
